# 🎯 05 — Backpropagation

**Date:** 05 June 2026  
**Stage:** Deep Learning  
**Module:** Backpropagation  
**Framework:** PyTorch & NumPy

---

## 1. What You Will Learn

By the end of this module, you should understand:

* What backpropagation actually is & why neural networks need it
* Computational graphs, derivatives, and the **Chain Rule**
* Forward pass vs. Backward pass with manual numerical calculations
* From-scratch NumPy implementation of forward and backward passes
* PyTorch Autograd (`.backward()`, `.grad`, `requires_grad=True`)
* The standard PyTorch optimization loop (`zero_grad()`, `optimizer.step()`)
* Gradient accumulation, `detach()`, and `torch.no_grad()`
* Vanishing and Exploding gradients, Recurrent vulnerabilities, and Gradient Clipping
* Debugging checklists, common mistakes, and interview practice questions

## 2. Fundamentals: What & Why Backpropagation?

**Backpropagation** is the algorithm used to efficiently calculate how much each model parameter contributed to the final loss.

```text
Input ──> Forward Pass ──> Prediction ──> Loss
                                            │
Updated Weights <── Optimizer <── Gradients <── Backpropagation
```

> **Key Distinction:**  
> • **Backpropagation** calculates the gradients ($\frac{\partial L}{\partial w}$).  
> • **Optimizer** uses gradients to update parameters ($w_{new} = w - \eta \frac{\partial L}{\partial w}$).

### The Chain Rule & Derivatives
A derivative measures sensitivity: how much $y$ changes when $x$ changes ($y = x^2 \implies \frac{dy}{dx} = 2x$).

In deep networks ($x \rightarrow a \rightarrow b \rightarrow L$), we calculate the gradient with respect to any input parameter by chaining local derivatives together:

$$\boxed{\frac{\partial L}{\partial x} = \frac{\partial L}{\partial b} \cdot \frac{\partial b}{\partial a} \cdot \frac{\partial a}{\partial x}}$$

## 3. Step-by-Step Manual Numerical Example

Let's trace a single neuron with activation and loss:
1. **Forward:** $z = wx + b$, $L = (z - y)^2$
2. **Given values:** $x = 2$, $w = 3$, $b = 1$, target $y = 5$
3. **Forward Pass Calculation:** $z = (3)(2) + 1 = 7$, $\text{Loss} = (7 - 5)^2 = 4$

### Local Derivatives & Backward Pass
- $\frac{\partial L}{\partial z} = 2(z - y) = 2(7 - 5) = 4$
- $\frac{\partial z}{\partial w} = x = 2 \implies \frac{\partial L}{\partial w} = 4 \times 2 = \mathbf{8}$
- $\frac{\partial z}{\partial b} = 1 \implies \frac{\partial L}{\partial b} = 4 \times 1 = \mathbf{4}$

With learning rate $\eta = 0.1$:
- $w_{new} = 3 - (0.1)(8) = \mathbf{2.2}$
- $b_{new} = 1 - (0.1)(4) = \mathbf{0.6}$

## 4. Pure NumPy Implementation From Scratch

Below is the exact code implementing the manual forward pass, analytical chain-rule backward pass, and parameter updates.

In [ ]:
import numpy as np

# Inputs & Target
x = 2.0
w = 3.0
b = 1.0
y = 5.0
learning_rate = 0.1

# --- 1. Forward Pass ---
z = w * x + b
loss = (z - y) ** 2
print(f"[NumPy Forward] z: {z:.2f}, Loss: {loss:.2f}")

# --- 2. Backward Pass (Chain Rule) ---
dL_dz = 2.0 * (z - y)
dL_dw = dL_dz * x
dL_db = dL_dz * 1.0

print(f"[NumPy Backward] dL/dz: {dL_dz:.2f}, dL/dw: {dL_dw:.2f}, dL/db: {dL_db:.2f}")

# --- 3. Parameter Update ---
w_new = w - learning_rate * dL_dw
b_new = b - learning_rate * dL_db
print(f"[NumPy Updated Params] w: {w_new:.2f}, b: {b_new:.2f}")

## 5. PyTorch Autograd Equivalence

Now let's reproduce the exact calculation using PyTorch automatic differentiation engine (`autograd`).

In [ ]:
import torch

# Define inputs and trainable parameters
x_t = torch.tensor(2.0)
w_t = torch.tensor(3.0, requires_grad=True)
b_t = torch.tensor(1.0, requires_grad=True)
y_t = torch.tensor(5.0)

# Forward pass
z_t = w_t * x_t + b_t
loss_t = (z_t - y_t) ** 2

# Backward pass
loss_t.backward()

print(f"[PyTorch Autograd] w.grad: {w_t.grad.item():.2f}")  # Expect 8.0
print(f"[PyTorch Autograd] b.grad: {b_t.grad.item():.2f}")  # Expect 4.0

## 6. The Standard PyTorch Training Step & `zero_grad()`

PyTorch accumulates gradients by default. Therefore, every iteration must follow this canonical structure:

```text
optimizer.zero_grad() ──> Forward Pass ──> Loss Computation ──> loss.backward() ──> optimizer.step()
```

In [ ]:
import torch.nn as nn
import torch.optim as optim

# Define Linear Model (1 input feature -> 1 output feature)
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

x_batch = torch.tensor([[2.0]])
y_batch = torch.tensor([[5.0]])

# Canonical Training Step
optimizer.zero_grad()       # 1. Clear previous gradients
prediction = model(x_batch) # 2. Forward pass
loss = loss_fn(prediction, y_batch) # 3. Compute loss
loss.backward()             # 4. Backpropagation (autograd graph traversal)
optimizer.step()            # 5. Parameter update step

print(f"Training step finished. Loss value: {loss.item():.4f}")

## 7. Gradient Accumulation, `torch.no_grad()`, and `detach()`

### Gradient Accumulation
When mini-batch size is constrained by memory (e.g., GPU memory limits), we can accumulate gradients over multiple smaller micro-batches before calling `optimizer.step()`.

### Inference & Evaluation Modes
- `torch.no_grad()` / `torch.inference_mode()` disables autograd tracking to accelerate inference and conserve memory.
- `model.eval()` turns off layers like Dropout/BatchNorm, but **does not** disable gradient tracking by itself.
- `y = x.detach()` returns a new tensor detached from the computation graph.

In [ ]:
# --- 1. Gradient Accumulation Example ---
accum_model = nn.Linear(2, 1)
accum_optimizer = optim.SGD(accum_model.parameters(), lr=0.01)
criterion = nn.MSELoss()

accum_optimizer.zero_grad()
accumulation_steps = 4

for step in range(accumulation_steps):
    x_sub = torch.randn(4, 2)
    y_sub = torch.randn(4, 1)
    
    # Scale loss so total gradient matches full batch size equivalent
    sub_loss = criterion(accum_model(x_sub), y_sub) / accumulation_steps
    sub_loss.backward()
    print(f"Micro-batch {step+1} - Accumulated Grad Norm: {accum_model.weight.grad.norm().item():.4f}")

accum_optimizer.step()
accum_optimizer.zero_grad()

# --- 2. Inference & Detach Example ---
accum_model.eval()
with torch.no_grad():
    sample_input = torch.randn(1, 2)
    inference_out = accum_model(sample_input)
    print("Inference Output (No Grad):", inference_out)

## 8. Gradient Pathology & Inspection: Clipping & Debugging

- **Vanishing Gradients:** Local gradients $< 1$ multiplied across deep layers cause gradients to approach $0$.
- **Exploding Gradients:** Local gradients $> 1$ multiplied across deep layers cause parameter instability or `NaN`s.
- **Gradient Clipping:** Prevents exploding gradients by capping maximum gradient norms (`nn.utils.clip_grad_norm_`).

In [ ]:
# Deep network gradient inspection and clipping demonstration
deep_net = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 20),
    nn.ReLU(),
    nn.Linear(20, 1)
)

dummy_input = torch.randn(1, 10)
dummy_output = deep_net(dummy_input)
dummy_output.backward()

print("=== Layer-wise Gradient Magnitude Inspection ===")
for name, param in deep_net.named_parameters():
    if param.grad is not None:
        print(f"{name:<20} | Mean |Grad|: {param.grad.abs().mean().item():.6f}")

# Apply Gradient Clipping
nn.utils.clip_grad_norm_(deep_net.parameters(), max_norm=1.0)
print("\n✅ Successfully applied gradient clipping (max_norm=1.0).")

## 9. Mini Experiment: Full Optimization Loop

Let's run a multi-epoch optimization loop to verify that backpropagation and gradient descent reduce the MSE loss over time.

In [ ]:
import matplotlib.pyplot as plt

# Synthetic dataset
torch.manual_seed(42)
X = torch.randn(100, 2)
true_w = torch.tensor([[1.5], [-2.0]])
Y = X @ true_w + 0.5

# Model, Loss, Optimizer
exp_model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)
exp_loss_fn = nn.MSELoss()
exp_optimizer = optim.Adam(exp_model.parameters(), lr=0.05)

loss_history = []

for epoch in range(100):
    exp_optimizer.zero_grad()
    preds = exp_model(X)
    l = exp_loss_fn(preds, Y)
    l.backward()
    exp_optimizer.step()
    loss_history.append(l.item())

print(f"Initial Loss: {loss_history[0]:.4f} ---> Final Loss: {loss_history[-1]:.4f}")

## 10. Summary & Interview Practice Questions

### Key Checklist
- [x] **Backpropagation vs Optimizer:** Backprop computes $\frac{\partial L}{\partial w}$, Optimizer applies update rule.
- [x] **PyTorch Order:** `zero_grad()` $\rightarrow$ `forward` $\rightarrow$ `loss` $\rightarrow$ `backward()` $\rightarrow$ `step()`.
- [x] **Gradient Accumulation:** Call `backward()` multiple times before `optimizer.step()` and `zero_grad()`.
- [x] **Vanishing/Exploding Gradients:** Caused by chain-rule products of deep layer derivatives; mitigated via ReLU, skip connections, and gradient clipping.

### Top Interview Questions
1. *Why can't we use numerical differentiation (finite differences) to train neural networks?*
   - Numerical differentiation requires $O(N)$ forward passes per step for $N$ parameters. Backpropagation computes all gradients in a single backward pass ($O(1)$ scaling relative to forward cost).
2. *What happens if you forget `optimizer.zero_grad()`?*
   - Gradients accumulate across batches. The update step will use sum of old and current gradients, causing incorrect updates.
3. *What is the difference between `model.eval()` and `torch.no_grad()`?*
   - `model.eval()` changes operational mode for layers like Dropout/BatchNorm. `torch.no_grad()` disables tracking in autograd engine to save memory.